# 00 — UGAP Colab Setup

Run this notebook first, every session (Colab disks are ephemeral).
It:
1. Mounts Google Drive (so checkpoints/data persist across sessions)
2. Clones this repo (if not already present) and installs requirements
3. Clones DUSt3R and GraspNet-baseline
4. Downloads the DUSt3R checkpoint and the GraspNet pretrained checkpoint

**Runtime:** Runtime > Change runtime type > GPU (T4 is enough for this project).


In [ ]:
# 1. Mount Google Drive so checkpoints/data survive session resets
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ugap_data'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive mounted. Persistent data will live at:", DRIVE_ROOT)


In [ ]:
# 2. Clone the UGAP repo (skip if you uploaded it directly instead)
REPO_URL = "https://github.com/<your-username>/UGAP.git"  # <-- fill in once pushed

import os
if not os.path.exists('UGAP'):
    !git clone {REPO_URL}
%cd UGAP
!pip install -q -r requirements.txt


In [ ]:
# 3. Clone DUSt3R (reconstruction backbone)
import sys, os
if not os.path.exists('dust3r'):
    !git clone --recursive https://github.com/naver/dust3r
sys.path.append('dust3r')
!pip install -q -r dust3r/requirements.txt
print("DUSt3R ready.")


In [ ]:
# 4. Clone GR-ConvNet reference repo (primary grasp detector -- Cornell/Jacquard)
import sys, os
if not os.path.exists('robotic-grasping'):
    !git clone https://github.com/skumra/robotic-grasping
sys.path.append('robotic-grasping')
!pip install -q -r robotic-grasping/requirements.txt
print("GR-ConvNet (robotic-grasping) ready.")


In [ ]:
# 5. Download model checkpoints
# Symlink into Drive so we don't re-download every session.
import os
CKPT_DRIVE = os.path.join(DRIVE_ROOT, 'checkpoints')
os.makedirs(CKPT_DRIVE, exist_ok=True)
if not os.path.islink('checkpoints'):
    if os.path.exists('checkpoints'):
        !rm -rf checkpoints
    os.symlink(CKPT_DRIVE, 'checkpoints')

!bash scripts/download_models.sh


### GR-ConvNet pretrained checkpoint (manual step)

Pretrained weights for both Cornell and Jacquard are distributed with the
reference repo -- check its README / releases page for the current
download link, then either:

1. Download manually and upload to
   `checkpoints/grconvnet/<checkpoint_name>` (e.g. `cornell_rgbd_iou_0.96`),
   or directly into `ugap_data/checkpoints/grconvnet/` in your Drive.
2. Or, if you'd rather train it yourself (fast -- Cornell is small):
   ```
   !cd robotic-grasping && python train_network.py \
       --dataset cornell \
       --dataset-path ../data/cornell \
       --description training_cornell
   ```
   This trains in well under an hour on a single Colab T4 GPU.

### GraspNet-baseline checkpoint (OPTIONAL / legacy)

Only needed if you later want the stretch-goal comparison against the
full 6-DoF GraspNet pipeline (see `scripts/download_models.sh`). Skip
this by default.


In [ ]:
# 6. Sanity check: everything importable
import torch
print("CUDA available:", torch.cuda.is_available())

import sys
sys.path.append('.')
from ugap.reconstruction import DUSt3RWrapper
from ugap.uncertainty import EvidentialUncertaintyHead
from ugap.nbv import NextBestViewPlanner
from ugap.grasping import GRConvNetWrapper
print("UGAP package imports OK.")
